# AI CV Reviewer: step by step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarAfet/sdaia-agentic-ai-cv-reviewer/blob/main/notebooks/run_in_colab.ipynb)

This notebook runs one CV through the system, one step per cell:

1. Install → 2. API key → 3. Choose a CV → 4. Intake → 5. Security →
6. **Parser agent** → 7. Skill-match tool → 8. **Matcher agent** → 9. **Reviewer agent** →
10. HR decision → 11. Report → 12. All CVs at once → 13. Download

It asks for no input except, as a fallback, the API key in step 2. The CVs and HR decisions come from `data/colab_cvs.json`.
Use **Runtime → Run all**.

**Before you start:** get a free key at <https://console.groq.com/keys>. In Colab, click the key
icon (Secrets) in the left panel, add `GROQ_API_KEY`, and turn on "Notebook access".
If Secrets do not work (for example, in VS Code), step 2 asks for the key once.

## Step 1. Install

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = "/content/sdaia-agentic-ai-cv-reviewer"
if not os.path.isdir(ROOT):
    !git clone -q --depth 1 https://github.com/OmarAfet/sdaia-agentic-ai-cv-reviewer.git {ROOT}
os.chdir(ROOT)
!pip install -q -e .
sys.path.insert(0, f"{ROOT}/src")   # the running kernel does not see the new install path

def show(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

## Step 2. API key

The cell looks for the key in this order:
1. an environment variable `GROQ_API_KEY` that is already set,
2. Colab Secrets (works only in the Colab web page, not in VS Code or other front ends),
3. a hidden prompt, only if 1 and 2 fail.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        print("Key from Colab Secrets.")
    except Exception as e:   # TimeoutException outside the Colab web page, SecretNotFoundError, ...
        print(f"Colab Secrets not available ({type(e).__name__}).")
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

from cv_reviewer.config import Settings
from cv_reviewer.llm import GroqLLM
from cv_reviewer.observability import Tracer

settings = Settings()
tracer = Tracer("colab-step-by-step", settings.log_dir)
llm = GroqLLM(settings, tracer)
print("Ready.")

## Step 3. Choose a CV

Change `CASE` to any id in the list, then run the next cells again.

In [ ]:
cases = {c["id"]: c for c in json.loads(Path("data/colab_cvs.json").read_text())["cases"]}
for c in cases.values():
    print(f"{c['id']:16} {c['job_id']:17} {c['label']}")

CASE = "analyst_strong"

case = cases[CASE]
if "text" in case:
    cv_path = Path(f"/content/{CASE}.txt")
    cv_path.write_text(case["text"])
else:
    cv_path = Path(case["file"])

state = {"run_id": f"colab-{CASE}", "job_id": case["job_id"], "status": "running", "errors": []}

def stopped():
    if state["status"] != "running":
        print(f"Skipped: status is '{state['status']}'. Steps 10 and 11 still run.")
        return True
    return False

## Step 4. Intake

Checks the file type and size, reads the CV, finds hidden text, and looks up the job.

In [ ]:
from cv_reviewer.guardrails.validation import validate_file, validate_text
from cv_reviewer.tools.file_reader import read_cv
from cv_reviewer.tools.job_db import JobDB

state["job"] = JobDB(settings.jobs_file).get_job(case["job_id"])
doc = read_cv(cv_path)
errors = validate_file(cv_path, settings) + validate_text(doc.visible_text, settings)

if errors:
    state["status"], state["errors"] = "invalid", errors
state["visible_text"], state["hidden_text"] = doc.visible_text, doc.hidden_text

print("Status:", state["status"])
print("Errors:", errors or "none")
print("Hidden text found:", bool(doc.hidden_text))

## Step 5. Security

Hides personal data **before** any model sees the CV, then scans for prompt injection
(regex rules + Prompt Guard 2 + gpt-oss-safeguard).

In [ ]:
from cv_reviewer.guardrails.pii import mask_pii
from cv_reviewer.guardrails.injection import detect_injection

if not stopped():
    masked, counts = mask_pii(doc.visible_text)
    masked_hidden, _ = mask_pii(doc.hidden_text)
    inj = detect_injection(masked, masked_hidden, doc.zero_width_chars,
                           classifier=llm.injection_score, threshold=settings.injection_threshold,
                           policy_classifier=llm.policy_violation)
    state.update(masked_text=masked, pii_counts=counts, injection=inj)
    if inj["detected"]:
        state["status"] = "quarantined"

    print("Masked:", counts)
    print("Injection found:", inj["detected"], "| rules:", [h["rule"] for h in inj["rule_hits"]])
    print("\nWhat the agents will see:\n")
    print(masked[:600])

## Step 6. Parser agent

Reads the masked CV and extracts facts only: skills, years, roles, education.

In [ ]:
from cv_reviewer.agents.parser import run_parser

if not stopped():
    state["profile"] = run_parser(llm, state["masked_text"]).model_dump()
    show(state["profile"])

## Step 7. Skill-match tool

A fixed formula (no AI) that compares the CV skills and years with the job. The Matcher starts from this score.

In [ ]:
from cv_reviewer.tools.skill_match import skill_match

if not stopped():
    p = state["profile"]
    state["tool_result"] = skill_match(state["job"], p["skills"], p["total_years_experience"],
                                       state["masked_text"])
    show(state["tool_result"])

## Step 8. Matcher agent

Scores the fit (0–100) with evidence from the CV, lists gaps, and writes feedback for the candidate.

In [ ]:
from cv_reviewer.agents.matcher import run_matcher

if not stopped():
    state["match"] = run_matcher(llm, state["profile"], state["job"], state["tool_result"]).model_dump()
    show(state["match"])

## Step 9. Reviewer agent

Checks the Matcher's answer: claims without evidence, bias, and a score far from the tool score.
If it rejects the answer, the Matcher revises it (maximum 2 times), then a person decides.

In [ ]:
from cv_reviewer.agents.reviewer import run_reviewer

if not stopped():
    state["review_history"] = []
    for revision in range(settings.max_revisions + 1):
        review, _ = run_reviewer(llm, state["profile"], state["job"], state["tool_result"],
                                 state["match"], settings.max_score_drift)
        state["review"] = review.model_dump()
        state["review_history"].append({"revision": revision, "score": state["match"]["score"],
                                        "approved": review.approved, "issues": review.issues})
        print(f"Round {revision + 1}: score {state['match']['score']} ->",
              "approved" if review.approved else f"rejected: {review.issues}")
        if review.approved:
            break
        if revision < settings.max_revisions:
            state["match"] = run_matcher(llm, state["profile"], state["job"], state["tool_result"],
                                         previous=state["match"], critique=state["review"]).model_dump()
    else:
        state["status"] = "needs_human_review"

## Step 10. HR decision

The AI only proposes. Score ≥ 70 → shortlist; below → reject; quarantined or unresolved → hold.
The person's decision comes from the `hr` field in `data/colab_cvs.json` (`null` = accept the proposal).

In [ ]:
if state["status"] == "running":
    state["status"] = "proposed_shortlist" if state["match"]["score"] >= settings.shortlist_threshold else "proposed_reject"

if state["status"] != "invalid":
    proposal = {"proposed_shortlist": "shortlist", "proposed_reject": "reject"}.get(state["status"], "hold")
    hr = case.get("hr") or {"decision": proposal, "by": "pre-made (accept AI proposal)", "note": ""}
    state["hr_decision"] = {**hr, "proposal": proposal, "overridden": hr["decision"] != proposal}
    show(state["hr_decision"])
else:
    print("Invalid input: no decision needed.")

## Step 11. Report

In [ ]:
from IPython.display import Markdown, display
from cv_reviewer.report import build_report

report, path = build_report(state, tracer, settings)
display(Markdown(report))

## Step 12. All CVs at once

The same steps as above, run by the LangGraph workflow for every case in the JSON file.
The free Groq tier limits requests per minute, so this takes a few minutes.

In [ ]:
import pandas as pd
from cv_reviewer.pipeline import review_cv

rows = []
for c in cases.values():
    path = Path(c["file"]) if "file" in c else Path(f"/content/{c['id']}.txt")
    if "text" in c:
        path.write_text(c["text"])
    approve = lambda payload, c=c: c.get("hr") or {"decision": payload["proposal"], "by": "pre-made", "note": ""}
    print("reviewing", c["id"], "...", flush=True)
    st = review_cv(path, c["job_id"], approver=approve)["state"]
    hr = st.get("hr_decision") or {}
    rows.append({"case": c["id"], "job": c["job_id"], "status": st["status"],
                 "score": (st.get("match") or {}).get("score", "-"),
                 "AI proposal": hr.get("proposal", "-"), "HR decision": hr.get("decision", "-")})
pd.DataFrame(rows)

## Step 13. Download the reports and logs

In [ ]:
from google.colab import files
!zip -qr /content/cv_reviewer_output.zip reports logs
files.download("/content/cv_reviewer_output.zip")